# LangChainについて

## LangChainとは

LangChainとは、大規模言語モデル（LLM）を活用したアプリケーション開発を支援するオープンソースフレームワークのこと。

### 注意点
- サンプルコードはGemini APIの無料枠を利用している
  - **無料枠はモデルの学習に利用される**
- LangChainは以下などの理由からあまり推奨されていないらしい..
  - 高度に抽象化されており、知識の応用が効かない(LangChain独特の知識)
  - 学習コストが高い
  - 性能懸念がある
  - RAGの精度を上げたい場合は結局SDKを使った方が良い

### 主な機能・コンポーネント

LangChainの主な機能・コンポーネントについて記載する。

#### プロンプトテンプレート機能

プロンプトをテンプレート化する機能。  
利用例を以下に示す。

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """以下の料理のレシピを考えてください
    料理名: {dish}"""
)

prompt_value = prompt.invoke({"dish": "カレー"})
print(prompt_value)

text='以下の料理のレシピを考えてください\n    料理名: カレー'


#### Output parser

文字列、JSON、Pythonオブジェクトなどの出力形式を提供する機能。  
利用例を以下に示す。

In [ ]:
# 不要なパッケージのアンインストール（他のパッケージと依存関係がぶつかる）
!pip uninstall -y google-adk opentelemetry-exporter-otlp-proto-http
# 必要なパッケージのインストール
!pip install -q -U langchain-google-genai

##### 文字列を出力する場合の例

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザが入力した料理のレシピを考えてください。"),
        ("human", "{dish}")
    ]
)

# Geminiの無料枠を使う
model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=userdata.get("GEMINI_API_KEY") # 事前にシークレットに登録しておく
    )

output_parser = StrOutputParser()

prompt_value = prompt.invoke({"dish": "カレー"})
ai_message = model.invoke(prompt_value)
output = output_parser.invoke(ai_message)

print(output)

はい、承知いたしました！
市販のカレールーを使った、ご家庭で作りやすい定番カレーのレシピをご紹介します。
具材は鶏肉と定番野菜ですが、お好みの肉や野菜でアレンジしてくださいね。

---

## 我が家の定番！ごろごろ野菜とチキンの絶品カレー

市販のカレールーを使えば、誰でも簡単に美味しいカレーが作れます。玉ねぎをしっかり炒めるのがコクを出すポイント！

### 材料（4人分）

*   **鶏もも肉**：1枚（約300g）
*   **玉ねぎ**：1個（大）
*   **じゃがいも**：2個
*   **人参**：1本
*   **サラダ油**：大さじ1
*   **水**：800ml（※カレールーの箱の表示に従ってください）
*   **市販のカレールー**：1箱（4皿分）
*   **ご飯**：適量

**【お好みで】**
*   にんにくチューブ、しょうがチューブ：各2cm
*   隠し味（ケチャップ、ウスターソース、インスタントコーヒー、はちみつなど）：少量

### 作り方

1.  **下準備をする**
    *   鶏もも肉は一口大に切る。
    *   玉ねぎはくし切りにする。
    *   じゃがいもと人参は乱切りにする。じゃがいもは水に5分ほどさらし、アク抜きをしてから水気を切る。

2.  **具材を炒める**
    *   厚手の鍋にサラダ油を熱し、鶏肉を皮目から入れて焼き色がつくまで炒める。
    *   玉ねぎを加えて、しんなり透明になるまでじっくり炒める。（お好みでにんにく、しょうがもここで加える）
    *   人参、じゃがいもを加えて、全体に油が回るように軽く炒め合わせる。

3.  **煮込む**
    *   水を加えて強火にし、沸騰したらアクを丁寧に取り除く。
    *   蓋をして弱火で15～20分、野菜が柔らかくなるまで煮込む。

4.  **ルーを加える**
    *   一度火を止め、カレールーを割り入れて溶かす。
    *   再び弱火にかけ、とろみがつくまで時々鍋底から混ぜながら5～10分煮込む。
    *   お好みで隠し味を少量ずつ加えて味を調える。

5.  **盛り付け**
    *   器にご飯を盛り、熱々のカレーをたっぷりかけて召し上がれ！

### 美味しく作るポイント



##### Pythonオブジェクトを出力する場合の例

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="step to make the dish")

output_parser = PydanticOutputParser(pydantic_object=Recipe)
format_instructions = output_parser.get_format_instructions()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", "ユーザが入力した料理のレシピを考えてください。\n\n"
            "{format_instructions}",
        ),
        ("human", "{dish}"),
    ]
)
prompt_with_format_instructions = prompt.partial(
    format_instructions=format_instructions
)
prompt_value = prompt_with_format_instructions.invoke({"dish": "カレー"})

# Geminiを使う
model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=userdata.get("GEMINI_API_KEY") # 事前にシークレットに登録しておく
    )

ai_message = model.invoke(prompt_value)
recipe = output_parser.invoke(ai_message)

print(type(recipe))
print(recipe)

<class '__main__.Recipe'>
ingredients=['肉（鶏肉、豚肉、牛肉など）: 200-300g', '玉ねぎ: 1個', '人参: 1本', 'じゃがいも: 2個', 'カレールー: 1箱（市販品、4-6皿分）', '水: ルーの箱の表示に従う（通常600-800ml）', 'サラダ油: 大さじ1'] steps=['肉、玉ねぎ、人参、じゃがいもを一口大に切ります。', '厚手の鍋にサラダ油を熱し、肉を炒めます。肉の色が変わったら、玉ねぎを加えてしんなりするまで炒めます。', '人参、じゃがいもを加えて軽く炒め合わせます。', '水を加え、沸騰したらアクを取り、蓋をして野菜が柔らかくなるまで弱火で15-20分煮込みます。', '一度火を止め、カレールーを割り入れて溶かします。', '再び弱火にかけ、とろみがつくまで時々混ぜながら5-10分煮込みます。', '器にご飯を盛り、カレーをかけてお召し上がりください。']


#### Memory

会話履歴を保存するための機能。  
メモリ上やデータベースに保存できる。

session_id を指定することで、会話をユーザ単位で分離できる。

#### LangChain Expression Language(LCEL)

処理の連鎖(chain)を実現するための機能。

LCELを使わない場合以下のように記述する。

```
# プロンプト生成
prompt_value = prompt.invoke({"dish": "カレー"})
# LLMの実行
ai_message = model.invoke(prompt_value)
# 出力の準備と整形
output_parser = StrOutputParser()
output = output_parser.invoke(ai_message)
```

LCELを使うと以下のように | を使って記述できる。

```
chain = prompt | model | StrOutputParser()
output = chain.invoke({"dish": "カレー"})
```

利用例を以下に示す。

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザが入力した料理のレシピを考えてください。"),
        ("human", "{dish}")
    ]
)

# Geminiを使う
model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=userdata.get("GEMINI_API_KEY") # 事前にシークレットに登録しておく
    )

chain = prompt | model | StrOutputParser()
output = chain.invoke({"dish": "カレー"})
print(output)

はい、承知いたしました！
市販のカレールーを使った、ご家庭で作りやすい定番カレーのレシピをご紹介します。
具材は鶏肉と定番野菜ですが、お好みの肉や野菜でアレンジしてくださいね。

---

## 我が家の定番！ごろごろ野菜とチキンの絶品カレー

市販のカレールーを使えば、誰でも簡単に美味しいカレーが作れます。玉ねぎをしっかり炒めるのがコクを出すポイント！

### 材料（4人分）

*   **鶏もも肉**：1枚（約300g）
*   **玉ねぎ**：1個（大）
*   **じゃがいも**：2個
*   **人参**：1本
*   **サラダ油**：大さじ1
*   **水**：800ml（※カレールーの箱の表示に従ってください）
*   **市販のカレールー**：1箱（4皿分）
*   **ご飯**：適量

**【お好みで】**
*   にんにくチューブ、しょうがチューブ：各2cm
*   隠し味（ケチャップ、ウスターソース、インスタントコーヒー、はちみつなど）：少量

### 作り方

1.  **下準備をする**
    *   鶏もも肉は一口大に切る。
    *   玉ねぎはくし切りにする。
    *   じゃがいもと人参は乱切りにする。じゃがいもは水に5分ほどさらし、アク抜きをしてから水気を切る。

2.  **具材を炒める**
    *   厚手の鍋にサラダ油を熱し、鶏肉を皮目から入れて焼き色がつくまで炒める。
    *   玉ねぎを加えて、しんなり透明になるまでじっくり炒める。（お好みでにんにく、しょうがもここで加える）
    *   人参、じゃがいもを加えて、全体に油が回るように軽く炒め合わせる。

3.  **煮込む**
    *   水を加えて強火にし、沸騰したらアクを丁寧に取り除く。
    *   蓋をして弱火で15～20分、野菜が柔らかくなるまで煮込む。

4.  **ルーを加える**
    *   一度火を止め、カレールーを割り入れて溶かす。
    *   再び弱火にかけ、とろみがつくまで時々鍋底から混ぜながら5～10分煮込む。
    *   お好みで隠し味を少量ずつ加えて味を調える。

5.  **盛り付け**
    *   器にご飯を盛り、熱々のカレーをたっぷりかけて召し上がれ！

### 美味しく作るポイント



#### Document loader

様々なデータソースからドキュメントを読み込むための機能

| Document loader | 概要
| --- | --- |
| GitLoader | リポジトリからファイルを読み込む |
| ConfluenceLoader | Confluenceのページを読み込み |
| UnstructuredLoader | テキストファイル、パワーポイント、HTML、PDF、画像などのファイルを読み込む |
| DirectoryLoader | ディレクトリ内のファイルをUnstructuredLoaderなどで読み込む |
| S3DirectoryLoader | Amazon S3のバケットを指定してオブジェクトを読み込む |

GitLoaderの例を以下に示す。  
※ 「https://github.com/junit-team/junit-framework/tree/main/junit-jupiter-params」のJavaファイルを対象に取得

In [ ]:
# 必要なパッケージのインストール
!pip install -q -U langchain langchain-community

In [27]:
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str) -> bool:
  return "junit-jupiter-params" in file_path and file_path.endswith(".java")

loader = GitLoader(
    clone_url="https://github.com/junit-team/junit-framework",
    repo_path="./junit-framework",
    branch="main",
    file_filter=file_filter,
)

raw_docs = loader.load()

print(f"ファイル名: {raw_docs[0].metadata.get("file_path", "不明")}")
print(f"先頭30文字: {raw_docs[0].page_content[:50]}...")

ファイル名: junit-jupiter-params/src/main/java/module-info.java
先頭30文字: /*
 * Copyright 2015-2026 the original author or a...


#### Document transformer

Document loaderで読み込んだドキュメントに変換をかけるための機能。  
例えば、ドキュメントをある程度のチャンクに分割できる。

利用例を以下に示す。

In [4]:
# 必要なパッケージのインストール
!pip install -q langchain-text-splitters

チャンクサイズ1000で分割する

In [28]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

java_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JAVA,
    chunk_size=1000,  # チャンクの最大サイズ
    chunk_overlap=10  # チャンク間の重複サイズ
)

# コードの分割実行
docs = java_splitter.split_documents(raw_docs)
print(docs[:2])

[Document(metadata={'source': 'junit-jupiter-params/src/main/java/module-info.java', 'file_path': 'junit-jupiter-params/src/main/java/module-info.java', 'file_name': 'module-info.java', 'file_type': '.java'}, page_content='/*\n * Copyright 2015-2026 the original author or authors.\n *\n * All rights reserved. This program and the accompanying materials are\n * made available under the terms of the Eclipse Public License v2.0 which\n * accompanies this distribution and is available at\n *\n * https://www.eclipse.org/legal/epl-v20.html\n */\n\n/**\n * JUnit Jupiter extension for parameterized tests.\n *\n * @since 5.0\n */\nmodule org.junit.jupiter.params {\n\n\trequires static transitive org.apiguardian.api;\n\trequires static transitive org.jspecify;\n\n\trequires transitive org.junit.jupiter.api;\n\trequires transitive org.junit.platform.commons;\n\n\texports org.junit.jupiter.params;\n\texports org.junit.jupiter.params.aggregator;\n\texports org.junit.jupiter.params.converter;\n\texp

#### Embedding model

ドキュメントの変換処理後に、テキストをベクトル化するための機能。

利用例を以下に示す。

In [29]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# 2. Gemini Embeddingモデルの設定
# 無料枠で利用可能な 'models/gemini-embedding-001' を使用
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT",  # ドキュメント登録用に最適化
    google_api_key=userdata.get("GEMINI_API_KEY") # 事前にシークレットに登録しておく
)

# ベクトル化の例
print(embeddings.embed_query("任意の文字列をベクトル化する"))

[-0.01880163, -0.004134218, 0.010412937, -0.08069568, 0.012144683, 0.023088612, 0.011634183, 0.012314894, 0.033910364, 0.013011293, -0.012972001, -0.016643818, 0.015450271, -0.001982464, 0.12756598, -0.016957378, -0.01448829, 0.010286367, -0.031819735, -0.03264455, -0.0027475546, 0.019534992, 0.011611354, -0.008880764, -0.021813009, -0.009944674, 0.01910922, -0.013392132, 0.01097993, -0.002694217, -0.007883235, 0.038075037, 0.021226484, 0.001848113, -0.016539313, -0.011276788, 0.022870665, 0.0033572053, -0.0014194768, 0.022207357, 0.00092740677, -0.019283608, -0.0026335616, -0.0057724724, 0.01918344, 0.031699233, 0.0010525365, -0.03378156, 0.01486828, 0.024402812, 0.006187059, -0.0052970923, -0.007584462, -0.2073528, 0.011541446, 0.009025763, -0.0048292, -0.016558073, 0.029263634, -0.01384695, 0.013448356, 0.02418689, -0.0047837757, -0.0022922216, 0.03465907, 0.0001767787, 0.02817125, 0.015764479, -0.012076494, -0.012894432, 0.0057427874, 0.0012700948, -0.012494577, -0.031474277, -0.02

#### Vector store

ベクトル化したドキュメントの保存先のこと。

LangChainと親和性のあるOSSであるChromaを利用した例を以下に示す。

In [ ]:
# 必要なパッケージのインストール
# 実行後セッションの再起動が必要
!pip install -q -U --force-reinstall langchain-chroma

In [32]:
from langchain_chroma import Chroma

# Vector storeに保存（複数ドキュメントを対象にリクエストすると無料枠の上限にひっかかるため対象を減らす）
db = Chroma.from_documents(docs[:50], embeddings)

# 保存先の確認
retriever = db.as_retriever()
context_docs = retriever.invoke("パラメータ化テストに利用するアノテーションを教えてください")
print(context_docs[0].page_content)

public @interface AfterParameterizedClassInvocation {

	/**
	 * Whether the arguments of the parameterized test class should be injected
	 * into the annotated method (defaults to {@code true}).
	 */
	boolean injectArguments() default true;

}


LLMを使ってVector Storeに問い合わせる。

In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に回答してください

文脈： """
{context}
"""

質問: {question}
''')

# Geminiを使う
model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=userdata.get("GEMINI_API_KEY") # 事前にシークレットに登録しておく
    )

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

output = chain.invoke("パラメータ化テストに利用するアノテーションを教えてください")
print(output)

文脈から読み取れるパラメータ化テストに利用するアノテーションは、`AfterParameterizedClassInvocation` です。
